In [10]:
import os
from pathlib import Path
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from dotenv import load_dotenv
load_dotenv()

import feedparser
import re
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, START, END

In [11]:
import os
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [12]:
from typing import Annotated, TypedDict, Optional, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage

def overwrite(old: Optional[any], new: Optional[any]):
    return new

class RSSState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    # Stores the raw articles before they are formatted
    raw_feed_data: Annotated[Optional[List[dict]], overwrite]
    # Reuse your existing Pydantic model for the final output
    files_to_create: Annotated[Optional[FilePackage], overwrite]

In [13]:
from pydantic import BaseModel, Field
from typing import List, Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Schema for Routing
class RouteDecision(BaseModel):
    destination: Literal["email", "code", "story"] = Field(description="The category of the request")

# Schema for Files
class FileContent(BaseModel):
    file_name: str = Field(description="Name of the file with extension")
    content: str = Field(description="The formatted text content. Use \\n for newlines.")

class FilePackage(BaseModel):
    files: List[FileContent] = Field(description="List of files to create")

# Initialize Parsers
route_parser = PydanticOutputParser(pydantic_object=RouteDecision)
file_parser = PydanticOutputParser(pydantic_object=FilePackage)
router_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a routing expert. Decide the destination based on user intent.\n{format_instructions}"),
    ("human", "{request}")
])

# Specialized Agent Prompt Template
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Generate high-quality content.\n"
               "CRITICAL: Use proper formatting, indentations, and '\\n' for new lines.\n"
               "{format_instructions}"),
    ("human", "{request}")
])

In [14]:
def fetch_news_node(state: RSSState):
    """Extracts RSS links and ensures unique article links are captured accurately."""
    user_message = state["messages"][-1].content
    
    # Regex to find all URLs
    url_pattern = r'https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+[/\w\.-]*(?:\?\S*)?'
    urls = re.findall(url_pattern, user_message)
    
    if not urls:
        urls = ["https://techcrunch.com/feed/"]
    
    all_articles = []
    for url in urls:
        feed = feedparser.parse(url)
        # Check for parsing errors
        if feed.bozo:
            continue
            
        for entry in feed.entries[:3]:
            # feedparser usually puts the permalink in entry.link
            # We use .get() to avoid attribute errors
            link = entry.get('link', url)
            
            all_articles.append({
                "article_link": link,
                "title": entry.get('title', 'No Title'),
                "summary": entry.get('summary', '')[:300] + "..."
            })
    
    return {
        "raw_feed_data": all_articles,
        "messages": [AIMessage(content=f"Fetched {len(all_articles)} entries from {len(urls)} sources.")]
    }

def format_news_node(state: RSSState):
    """Uses Placeholder IDs and Pydantic models to ensure 100% URL accuracy."""
    llm_json = llm.bind(response_format={"type": "json_object"})
    articles = state["raw_feed_data"]
    
    # 1. Map real URLs to simple IDs to hide them from the LLM's imagination
    news_input_context = ""
    url_map = {}
    
    for i, a in enumerate(articles):
        # We use a distinct pattern like [[REF_N]] which is easy for LLMs to repeat
        placeholder = f"[[REF_{i}]]"
        url_map[placeholder] = a['article_link']
        
        news_input_context += (
            f"ID: {placeholder}\n"
            f"TITLE: {a['title']}\n"
            f"SUMMARY: {a['summary']}\n\n"
        )
    
    # 2. Instruct the LLM to write the summary using ONLY the Placeholder ID
    request_instruction = (
        "You are a News Editor. Create a 'daily_briefing.txt' file. "
        "Summarize each news item provided below in exactly 1 sentence. "
        "At the end of each sentence, you MUST include the associated ID (e.g., [[REF_0]]) in parentheses. "
        "DO NOT write or invent actual URLs. Use the IDs exactly as provided.\n\n"
        f"DATA TO PROCESS:\n{news_input_context}"
    )
    
    prompt = agent_prompt.format_prompt(
        role="News Editor",
        request=request_instruction,
        format_instructions=file_parser.get_format_instructions()
    )
    
    # Invoke LLM and parse into your FilePackage model
    response = llm_json.invoke(prompt.to_messages())
    parsed_package: FilePackage = file_parser.parse(response.content)
    
    # 3. POST-PROCESSING: The "Golden Swap"
    # We iterate through the FileContent objects in the FilePackage
    for file_item in parsed_package.files:
        # Swap every placeholder found in the text with the actual URL from our map
        for placeholder, real_url in url_map.items():
            if placeholder in file_item.content:
                file_item.content = file_item.content.replace(placeholder, real_url)
    
    return {"files_to_create": parsed_package}
# --- Graph Construction ---

rss_builder = StateGraph(RSSState)

rss_builder.add_node("fetcher", fetch_news_node)
rss_builder.add_node("formatter", format_news_node)

rss_builder.add_edge(START, "fetcher")
rss_builder.add_edge("fetcher", "formatter")
rss_builder.add_edge("formatter", END)

rss_workflow = rss_builder.compile()


In [15]:
user_prompt = """
Please summarize the latest updates from these specific AI and Science feeds:
1. OpenAI: https://openai.com/news/rss.xml
2. Anthropic (Claude): https://www.anthropic.com/index.xml
3. NASA: https://www.nasa.gov/news-release/feed/
"""

print("--- Running RSS News Workflow with AI & NASA Feeds ---")

final_state = rss_workflow.invoke({
    "messages": [
        HumanMessage(content=user_prompt)
    ]
})

if final_state["files_to_create"]:
    for file in final_state["files_to_create"].files:
        print(f"FILENAME: {file.file_name}")
        print("-" * 30)
        print(file.content)

--- Running RSS News Workflow with AI & NASA Feeds ---
FILENAME: daily_briefing.txt
------------------------------
Tolan has developed a voice-first AI companion using GPT-5.1, enabling natural conversations with low-latency responses and real-time context reconstruction (https://openai.com/index/tolan)
ChatGPT Health is a new experience that connects users' health data and apps while prioritizing privacy and physician-informed design (https://openai.com/index/introducing-chatgpt-health)
OpenAI is now accepting applications for its Grove Cohort 2, a 5-week founder program offering API credits, early access to AI tools, and mentorship (https://openai.com/index/openai-grove)
The massive iceberg A-23A is disintegrating after a four-decade presence, with Meltwater playing a significant role in this development (https://science.nasa.gov/earth/earth-observatory/meltwater-turns-iceberg-a-23a-blue/)
NASA celebrated its Artemis II mission during the Houston Texans' Space City Day, giving fans a